**Quest 12: Desert Shower**

![Header image](./imgs/image_quest_12.svg)
[<img src="./imgs/instagram.svg" alt="My SVG" width="15" height="15"><small>_Monika Lipińska_</small>](https://www.instagram.com/monli_art/)

# Part I

### Story Section

The afternoon sun warms the faces of the weary knights in the tournament. The next round of challenges is scheduled to take place on the **Memory Stack Desert**, a flat surface of terrain stretching far into the horizon. Despite the late hour, it is still quite hot — though the sun will soon set.

The brave contestants arrive at a row of catapults. However, these are not the standard catapults used in other kingdoms. These machines have been engineered by the knightly Order, making them far more powerful!

Each catapult consists of **three segments stacked vertically**. From bottom to top, they are labeled:

- **A**
- **B**
- **C**

Each segment can launch projectiles independently, allowing up to **three simultaneous shots**.

Aiming is extremely precise: projectiles are unaffected by wind and travel at a constant speed determined by the **shooting power**, which can be set independently for each segment.

---

## Projectile Path

When launched, a projectile:

1. Travels diagonally upward at a 45° angle for _power_ segments
2. Then travels horizontally for _power_ segments
3. Then descends diagonally downward at 45° until it reaches the ground

### Example:

Projectile launched from **segment B** with **power 5**:

```
....................
......↗→→→→→........
.....↗......↘.......
....↗........↘......
...↗..........↘.....
.C↗............↘....
.B..............↘...
.A...............↘..
====================
```

Another example:  
Projectile launched from **segment C** with **power 3**:

```
....................
....................
....↗→→→............
...↗....↘...........
..↗......↘..........
.C........↘.........
.B.........↘........
.A..........↘.......
====================
```

---

## Targets

Each knight has several targets in front of them (your notes), marked with **T** on the map.  
A projectile destroys **only the first target it hits**.  
It is wise to destroy targets from the top downward to avoid unpredictable collapses.

---

## Ranking Value

Each shot has a ranking value:

$$
\text{segment number} \times \text{shooting power}
$$

Segment numbers:

- A → 1
- B → 2
- C → 3

Example:  
Shot from **B** with power **5** → ranking value = $2 \times 5 = 10$

Your task:  
Determine **which segments** and **what powers** to use to destroy all targets without wasting shots.  
Your score is the **sum of all ranking values**.

---

## Example Based on the Following Notes

```
.............
.C...........
.B......T....
.A......T.T..
=============
```

### Shot 1

Segment **C**, power **2**:

```
...↗→→.......
..↗...↘......
.C.....↘.....
.B......X....
.A......T.T..
=============
```

Ranking value: $3 \times 2 = 6$

---

### Shot 2

Segment **B**, power **2**:

```
.............
...↗→→.......
.C↗...↘......
.B.....↘.....
.A......X.T..
=============
```

Ranking value: $2 \times 2 = 4$

---

### Shot 3

Segment **A**, power **3**:

```
.............
....↗→→→.....
.C.↗....↘....
.B↗......↘...
.A........X..
=============
```

Ranking value: $1 \times 3 = 3$

---

### Total Ranking Value

$$
6 + 4 + 3 = 13
$$

---

## Question

**What is the ranking value of destroying your targets?**

```

```


In [51]:
from itertools import product
import re

from more_itertools import minmax

from util import hexcolor_str
from test_utilities import test

tests = [
    {
        "name": "Example Pat I",
        "notes": """
            .............
            .C...........
            .B......T....
            .A......T.T..
            =============
        """,
        "expected": 13,
    },
    {
        "name": "Example part II",
        "notes": """
            .............
            .C...........
            .B......H....
            .A......T.H..
            =============
        """,
        "expected": 22,
    },
]


class Note:
    SEGMENTS = {"A": (1, 1), "B": (2, 1), "C": (3, 1)}

    def __init__(self, s: str) -> None:
        self.grid = list(reversed(re.findall(r"\S+", s)))
        self.rows, self.cols = len(self.grid), len(self.grid[0])
        self._grid_map = {
            (r, c): self.grid[r][c]
            for r, c in product(range(self.rows), range(self.cols))
            if self.grid[r][c] != "."
        }

    def rank_point(self, row: int, col: int) -> int:
        # hit is on col projectile horizontal speed = t:
        # option 1: col < p
        t = col - 1
        for segment in range(1, 4):
            # ramp up
            #   1 + t = col => t = col - 1 if row + col - 1 in 1,2,3
            if segment + t == row:
                return segment * t

            # plateau
            # option 2: p < col <= 2p
            # p == row - row of segment =? row <=col<=2*col
            if row - segment > 0 and row - segment <= col <= 2 * (row - segment):
                return segment * (row - segment)

            # option 3: c > 2p
            # 3a row < row_segment
            if row < segment:
                p, mod = divmod(col - 1 + row - segment, 3)
                if mod == 0 and 1 + 3 * p + segment - row == col:
                    return segment * p

            # 3b row == row_segment
            if (col - 1) % 3 == 0:
                if row == segment:
                    return segment * (col - 1) // 3

            # 3c row > row_segment
            if row > segment and (col - 1 + row - segment) % 3 == 0:
                return segment * (col - 1 + row - segment) // 3

        raise ValueError()

    def total_ranking_value_v2(self) -> int:
        blocks = (
            (b, 1 if v == "T" else 2) for b, v in self._grid_map.items() if v in "TH"
        )

        total_ranking_value = 0
        for block, v in blocks:
            score = self.rank_point(*block)
            total_ranking_value += v * score

        return total_ranking_value

    def shoot(self, segment: str, power: int, already_hit: set[tuple[int, int]]) -> int:
        r, c = self.SEGMENTS[segment]

        for _ in range(power):
            r, c = r + 1, c + 1
            letter = self._grid_map.get((r, c), ".")
            if (r, c) not in already_hit and letter in "HT":
                hits += 1 if letter == "T" else 2
                already_hit.add((r, c))

        for _ in range(power):
            c += 1
            letter = self._grid_map.get((r, c), ".")
            if (r, c) not in already_hit and letter in "HT":
                hits += 1 if letter == "T" else 2
                already_hit.add((r, c))

        hits = 0

        while r > 1:
            r, c = r - 1, c + 1
            letter = self._grid_map.get((r, c), ".")
            if (r, c) not in already_hit and letter in "HT":
                hits += 1 if letter == "T" else 2
                already_hit.add((r, c))

        return hits

    def shoot_to_str(self, segment: str, power: int) -> str:
        r, c = self.SEGMENTS[segment]
        grid = {
            (r, c): self.grid[r][c]
            for r, c in product(range(self.rows), range(self.cols))
            if self.grid[r][c] != "."
        }

        for _ in range(power):
            r, c = r + 1, c + 1
            grid[(r, c)] = hexcolor_str("#FFFF00", "↗")

        for _ in range(power):
            c += 1
            grid[(r, c)] = hexcolor_str("#FFFF00", "→")

        while r > 1:
            r, c = r - 1, c + 1
            if grid.get((r, c), ".") == ".":
                grid[(r, c)] = hexcolor_str("#FFFF00", "↘")
            else:
                grid[(r, c)] = hexcolor_str("#FF0000", "X")

        grid[(r - 1, c + 1)] = hexcolor_str("#FF0000", "X")

        min_r, max_r = minmax(r for r, _ in grid.keys())
        min_c, max_c = minmax(c for _, c in grid.keys())

        return "\n".join(
            (
                "\n".join(
                    "".join(
                        grid.get((r, c), "." if r > 0 else "=")
                        for c in range(min_c, max_c + 1)
                    )
                    + f"{r:3d}"
                    for r in range(max_r, min_r - 1, -1)
                ),
                "".join(f"{c%10}" for c in range(min_c, max_c + 1)),
                "".join(
                    f"{c//10}" if c % 10 == 0 and c else " "
                    for c in range(min_c, max_c + 1)
                ),
            )
        )

    def total_ranking_value_v1(self) -> int:
        n = len(self.grid[0])
        already_hit = set()
        ranking_value = 0

        for seg, power in product("CBA", range(1, n)):
            hits = self.shoot(seg, power, already_hit)
            seg_nr = self.SEGMENTS[seg][0]
            ranking_value += hits * seg_nr * power

        return ranking_value

    def print_all_shots(self) -> None:
        n = len(self.grid[0]) // 3 + 1
        print(self)
        print()

        for seg, power in product(
            "CBA",
            range(1, n),
        ):
            print(f"segment={seg} and {power=}")
            print(self.shoot_to_str(seg, power))
            print()

    def __str__(self) -> str:
        return "\n".join("".join(row) for row in reversed(self.grid))


@test(tests=tests[:])
def part_I(notes: str) -> int:
    note = Note(notes)
    return note.total_ranking_value_v2()


Test Example Pat I passed, for part_I.
Test Example part II passed, for part_I.
Success


In [52]:
with open("../inputs/everybody_codes_e2024_q12_p1.txt") as f:
    notes1 = f.read()

print(f"Part I: {part_I(notes1)}")

Part I: 189


# Part II

In the second round, the rules remain unchanged. However, the targets are now the **ruins of a tower** (your notes), which once served as an observation post for the desert. Since so little happens there, the structure has been abandoned and is now useless.

Some of the ruin's blocks are made of **Hard Rocks (H)**, which require **two projectiles** to destroy.

---

### Example Based on the Following Notes

```
.............
.C...........
.B......H....
.A......T.H..
=============
```

The situation is almost the same as in the previous part, but some of the blocks (marked with **H**) need **two hits** instead of one.

The total ranking value for destroying all target blocks in this example is:

$$
(2 \times 6) + 4 + (2 \times 3) = 22
$$

---

### Question

**What is the ranking value of destroying all target blocks?**


In [53]:
with open("../inputs/everybody_codes_e2024_q12_p2.txt") as f:
    notes2 = f.read()

In [54]:
# def grid_str(grid: list[list[str]]) -> str:
#     return "\n".join("".join(row) for row in reversed(grid))


# grid = [list(l) for l in reversed(re.findall(r"\S+", notes2))]

# rows, cols = len(grid), len(grid[0])

# blocks = {
#     (r, c): grid[r][c] for r in range(rows) for c in range(cols) if grid[r][c] in "TH"
# }

# for r, c in blocks.keys():
#     grid[r][c] = "."

# for (r, c), v in blocks.items():
#     grid[r][c] = v

#     note = Note(grid_str(grid))

#     expected = note.total_ranking_value()
#     actual = note.all_intersects()

#     assert actual == expected, f"{actual=}, {expected=}, /n{note}"

#     grid[r][c] = "."

In [55]:
print(f"PartII: {part_I(notes2)}")

PartII: 20341


# Part III

Night has fallen over the desert. The stars twinkle softly above the Memory Stack, but some of them seem to be… moving. They grow brighter, larger — and then the knights realize the truth:

A **meteor shower** is descending toward the desert.

Fortunately, the meteors are falling far from inhabited lands. Even better, the catapult competition has been perfectly timed, because the final round now involves **defending against moving targets**.

All meteors travel at a constant speed, identical to the catapult projectiles, descending at a **45-degree angle** toward the knights.

Each projectile can destroy **exactly one** meteor.  
All knights collaborate on a single coordinated plan, and **multiple projectiles may be fired from the same segment at the same time**.

The knights record all meteor coordinates relative to segment **A** (your notes).  
For example, a meteor at coordinates `3 5` is:

- 3 segments to the right of A
- 5 segments above A

```
............
....#.......
............
............
.C..........
.B..........
.A..........
============
```

---

## Objective

The knights must shoot down the meteors **as high in the air as possible**, without wasting any projectiles.

If a meteor can be destroyed at the same altitude in more than one way, the knights choose the option with the **lower ranking score**.

- Shooting power is always an **integer**.
- Collisions can occur only at **discrete time steps**.
- Projectiles may be launched at **any discrete time**, not only at time 0.

---

## Example Based on the Following Notes

```
6 5
6 7
10 5
```

The initial situation:

```
.......#......
..............
.......#...#..
..............
..............
.C............
.B............
.A............
==============
```

### Meteor 1: coordinates `6 5`

It moves diagonally down-left each time step:

```
 time: 0
.......#......
...

 time: 1
......#.......
...

 time: 2
.....#........
...

 time: 3
....#.........
...

 time: 4
...#..........
...
```

A projectile from:

- **Segment C**, power **1**, fired at time **0**  
  hits the meteor at time **3**

But a projectile from:

- **Segment A**, power **2**, fired at time **0**  
  also hits it at time **3**, **with a lower ranking score (2)**

So the knights choose the shot from **A**.

---

### Meteor 2: coordinates `6 7`

It can be hit at altitude **4** by:

- Segment **B**, power **3** → ranking score **6**
- Segment **C**, power **2** → ranking score **6**

Both are equally good, so either is acceptable.

---

### Meteor 3: coordinates `10 5`

This meteor is barely reachable.  
It can only be destroyed by:

- Segment **C**, power **1**, at the lowest possible altitude  
  → ranking score **3**

---

### Total Ranking Value

$$
2 + 6 + 3 = 11
$$

This is the **best possible** total score for destroying all meteors.

---

## Another Example: When a Projectile Must Be Delayed

Meteor at:

```
5 5
```

Initial situation:

```
..............
..............
......#.......
..............
..............
.C............
.B............
.A............
==============
```

A shot from **A** at time 0 would collide at a **non-discrete** time — not allowed.

The knights must wait until time **1**, when the meteor has moved:

```
 time: 1
......↙.......
.....#........
```

Then a projectile from **A** with power **2** can destroy it at a valid discrete time.

Ranking value: **2**

---

## Question

**What is the lowest possible ranking score for the shots required to destroy all meteors at the highest altitudes possible?**

```

```


In [ ]:
from itertools import batched
from math import prod
from util import Str

tests = [
    {
        "name": "Example 1 Part III",
        "notes": """ 6 5 6 7 10 5 """,
        "expected": 11,
    },
    {
        "name": "Example 2 Part III delay needed",
        "notes": "5 5",
        "expected": 2,
    },
]


class NoteIII(Str):
    def __init__(self, notes: str) -> None:
        self.asteroids = list(batched(map(int, re.findall(r"\d+", notes)), n=2))

    def lowest_possible_ranking_score(self, x0: int, y0: int, debug: bool) -> int:
        if debug:
            start_message = f"astroid: {x0, y0}"

        intercepts = []
        # if x0 is odd there is no integer intercept,
        # for y0 this is different because of trapezoid
        # # because of the wanting to intercept the highest possible point,
        # # you take the first possible point.
        if x0 & 1 == 1:
            x0, y0 = x0 - 1, y0 - 1

        xi = x0 // 2
        yi = y0 - xi

        solutions = [[] for _ in range(3)]
        # must select the lowest segment:
        # # segment A store y intercept ie yi.
        # # intercept is on ramp up:
        # # In case of going straight through one of the segments
        # # one shot at power is xi is necessary. for getting the highest point.
        for d in range(3):
            # intercept on Ramp Up
            if x0 == y0 - d:
                intercepts.append((yi, d + 1, yi - d, "Ramp Up"))

            # intercept is on plateau:
            p = yi - d

            if p < xi <= 2 * p:
                intercepts.append((yi, d + 1, p, "Plateau"))

            # intercept is on ramp down

            p = (xi + (yi - d)) // 3
            yii = p + d - (xi - 2 * p)

            if xi > 2 * p and yii == yi:
                intercepts.append((yi, d + 1, p, "Ramp Down"))

        if debug:
            print(f"{start_message} => {intercepts=}")
            print(f"{max(intercepts, default=(0,0),key=lambda i: (i[0],-i[1]) )=}")

        return prod(max(intercepts, default=(0, 0), key=lambda i: (i[0], -i[1]))[1:-1])

    def lowest_possible_total_ranking_score(self, debug: bool = False) -> int:
        if debug:
            print(f"{len(self.asteroids)=}")

        total_ranking_value = 0
        for x0, y0 in self.asteroids:
            score = self.lowest_possible_ranking_score(x0, y0, debug)

            if debug:
                print(f"score for {x0,y0} = {score}\n")

            total_ranking_value += score

        return total_ranking_value


@test(tests=tests[:])
def part_III(notes: str) -> int:
    note = NoteIII(notes)
    return note.lowest_possible_total_ranking_score(debug=False)


Test Example 1 Part III passed, for part_III.
Test Example 2 Part III delay needed passed, for part_III.
Success


In [57]:
with open("../inputs/everybody_codes_e2024_q12_p3.txt") as f:
    notes3 = f.read()


print(f"Part III {part_III(notes3)}")

Part III 735057


![happy](./imgs/happy_quack.svg)
